# 🧥 MeshVTON — 3D Inference (Final)

**Girdi:** 2D kişi görseli + 3D kıyafet objesi (.obj)
**Çıktı:** kişi, kıyafeti giymiş halde

Kıyafet: database'den rastgele çekilebilir **veya** kendi `.obj`'unu yükleyebilirsin.

Akış: kişi → densepose + mask | 3D mesh → render (RGB+normal+depth) → eğitilmiş ControlNet3D → diffusion.

> Kurulum ~15-20 dk (detectron2 derleme + model indirme). Bir kere.

## 1️⃣ GPU + repolar

In [1]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON.git /content/MeshVTON
else:
    !cd /content/MeshVTON && git pull
if not os.path.exists('/content/IDM-VTON-official'):
    !git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON-official
print('✅ repolar hazır')

GPU: NVIDIA A100-SXM4-80GB
Cloning into '/content/MeshVTON'...
remote: Enumerating objects: 418, done.
remote: Counting objects: 100% (418/418), done.
remote: Compressing objects: 100% (271/271), done.
remote: Total 418 (delta 222), reused 330 (delta 134), pack-reused 0 (from 0)
Receiving objects: 100% (418/418), 2.58 MiB | 49.87 MiB/s, done.
Resolving deltas: 100% (222/222), done.
Cloning into '/content/IDM-VTON-official'...
remote: Enumerating objects: 1502, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 1502 (delta 103), reused 80 (delta 80), pack-reused 1238 (from 2)
Receiving objects: 100% (1502/1502), 22.19 MiB | 14.84 MiB/s, done.
Resolving deltas: 100% (407/407), done.
✅ repolar hazır


## 2️⃣ Kütüphaneler (+ detectron2 ~15dk) + Drive

In [2]:
os.environ["TOKENIZERS_PARALLELISM"]="false"; os.environ["FORCE_CUDA"]="1"
from google.colab import drive; drive.mount('/content/drive')
!pip install -q diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 peft==0.7.1
!pip install -q omegaconf opencv-python-headless einops av onnxruntime-gpu==1.17.1 fvcore iopath smplx trimesh
!pip install -q /content/drive/MyDrive/wheels/pytorch3d*.whl
!pip install -q -U onnxruntime
try:
    import detectron2
except Exception:
    !pip install 'git+https://github.com/facebookresearch/detectron2.git'
import detectron2; print('✅ detectron2', detectron2.__version__)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.20.3 which is incompatible.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.

## 3️⃣ Ön-işleme model dosyaları

In [3]:
import shutil
from huggingface_hub import hf_hub_download
base='/content/IDM-VTON-official/ckpt'
dp=f'{base}/densepose/model_final_162be9.pkl'; os.makedirs(os.path.dirname(dp),exist_ok=True)
if not os.path.exists(dp) or os.path.getsize(dp)<1e6:
    !wget -q https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl -O {dp}
for rf,d in {'humanparsing/parsing_atr.onnx':f'{base}/humanparsing/parsing_atr.onnx',
             'humanparsing/parsing_lip.onnx':f'{base}/humanparsing/parsing_lip.onnx',
             'openpose/ckpts/body_pose_model.pth':f'{base}/openpose/ckpts/body_pose_model.pth'}.items():
    if not os.path.exists(d) or os.path.getsize(d)<1e6:
        os.makedirs(os.path.dirname(d),exist_ok=True); shutil.copy(hf_hub_download('yisol/IDM-VTON',rf),d)
print('✅ ön-işleme modelleri hazır')

parsing_atr.onnx:   0%|          | 0.00/267M [00:00<?, ?B/s]

parsing_lip.onnx:   0%|          | 0.00/267M [00:00<?, ?B/s]

body_pose_model.pth:   0%|          | 0.00/209M [00:00<?, ?B/s]

✅ ön-işleme modelleri hazır


## 4️⃣ Pipeline + ControlNet3D (eğitilmiş) yükle

In [4]:
import sys, torch, glob
for m in list(sys.modules):
    if m=='src' or m.startswith('src.'): del sys.modules[m]
sys.path.insert(0,'/content/MeshVTON')
from src.idm_vton.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.idm_vton.unet_hacked_tryon import UNet2DConditionModel
from src.idm_vton.unet_hacked_garmnet import UNet2DConditionModel as RefUNet
from src.models.controlnet_3d import ControlNet3D
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection, CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer
from diffusers import DDPMScheduler, AutoencoderKL
B='yisol/IDM-VTON'; dt=torch.float16
unet=UNet2DConditionModel.from_pretrained(B,subfolder="unet",torch_dtype=dt)
uenc=RefUNet.from_pretrained(B,subfolder="unet_encoder",torch_dtype=dt)
vae=AutoencoderKL.from_pretrained(B,subfolder="vae",torch_dtype=dt)
ie=CLIPVisionModelWithProjection.from_pretrained(B,subfolder="image_encoder",torch_dtype=dt)
t1=CLIPTextModel.from_pretrained(B,subfolder="text_encoder",torch_dtype=dt)
t2=CLIPTextModelWithProjection.from_pretrained(B,subfolder="text_encoder_2",torch_dtype=dt)
k1=AutoTokenizer.from_pretrained(B,subfolder="tokenizer",use_fast=False)
k2=AutoTokenizer.from_pretrained(B,subfolder="tokenizer_2",use_fast=False)
sch=DDPMScheduler.from_pretrained(B,subfolder="scheduler")
pipe=TryonPipeline.from_pretrained(B,unet=unet,vae=vae,feature_extractor=CLIPImageProcessor(),
    text_encoder=t1,text_encoder_2=t2,tokenizer=k1,tokenizer_2=k2,scheduler=sch,image_encoder=ie,torch_dtype=dt)
pipe.unet_encoder=uenc; pipe=pipe.to('cuda'); pipe.unet_encoder.to('cuda')

cands=glob.glob('/content/drive/MyDrive/MeshVTON/checkpoints/meshvton/controlnet3d_final.pt') \
    + sorted(glob.glob('/content/drive/MyDrive/MeshVTON/checkpoints/meshvton/controlnet3d_*.pt')) \
    + sorted(glob.glob('/content/MeshVTON/checkpoints/meshvton/*.pt'))
assert cands, "❌ ControlNet3D checkpoint yok (Drive/MeshVTON/checkpoints)"
controlnet=ControlNet3D(conditioning_channels=9).to('cuda',dt).eval()
controlnet.load_state_dict(torch.load(cands[0],map_location='cuda'))
print('✅ pipeline + ControlNet3D yüklendi:', cands[0])

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytre

config.json: 0.00B [00:00, ?B/s]

diffusion_pytorch_model.bin:   0%|          | 0.00/12.0G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/8 [00:00<?, ?it/s]

✅ pipeline + ControlNet3D yüklendi: /content/drive/MyDrive/MeshVTON/checkpoints/meshvton/controlnet3d_final.pt


## 5️⃣ Ön-işleme + 3D render modülleri (paylaşılan conditioning yolu)

Giysi artık **kişinin tahmin edilen pozuna** göre render edilir (eğitimle birebir aynı
`build_conditioning_3d` yolu). Elle açı seçimi yok — kamera azimut'u SMPL-X `global_orient`'ten
otomatik türetilir (ön=0°, arka=180°).

In [ ]:
# --- 2D ön-işleme (IDM-VTON-official): parsing + openpose + densepose ---
sys.path.insert(0,'/content/IDM-VTON-official'); sys.path.insert(0,'/content/IDM-VTON-official/gradio_demo')
from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose
from utils_mask import get_mask_location
import apply_net
from detectron2.data.detection_utils import convert_PIL_to_numpy, _apply_exif_orientation
parsing_model=Parsing(0); openpose_model=OpenPose(0)

# --- 3D conditioning modülleri (eğitimle TEK kaynak: build_conditioning_3d) ---
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from src.modules.mesh_renderer import MeshRenderer
from src.modules.garment_draper import GarmentDraper
from src.modules.smplx_estimator import RealSMPLXEstimator
from src.modules.conditioning3d import build_conditioning_3d

renderer = MeshRenderer(image_size=512, device='cuda'); renderer.setup()
draper   = GarmentDraper().to('cuda').eval()
print('✅ render + drape modülleri hazır (gerçek SMPL-X pozu için sonraki hücre)')

## 5️⃣.B Gerçek SMPL-X poz tahmini (HMR2.0)

Kişinin gerçek pozunu/yönünü tahmin eder (ön/arka/yan). Gereksinimler:
- **SMPL-X neutral** modeli (kayıt gerektirir): `Drive/MeshVTON/smplx/SMPLX_NEUTRAL.npz`
- **4D-Humans** ilk çalıştırmada otomatik kurulur (~birkaç dk).

> Bu hücre olmadan giysi kişinin yönüne göre dönmez — eski sürümün asıl hatası buydu.

In [ ]:
# === Gerçek SMPL-X poz tahmini (4D-Humans / HMR2.0) ===
# Eski SimpleSMPLXRegressor EĞİTİLMEMİŞTİ → global_orient anlamsızdı (giysinin kişiye
# göre dönmemesinin kök nedeni). Artık görüntüden gerçek poz/yön tahmin ediyoruz.
import os, glob, shutil

# 1) SMPL-X neutral (renderer/draper) — Drive'dan
SMPLX_DIR = '/content/MeshVTON/checkpoints/pretrained/smplx'
os.makedirs(SMPLX_DIR, exist_ok=True)
for s in glob.glob('/content/drive/MyDrive/MeshVTON/smplx/SMPLX_NEUTRAL.*'):
    shutil.copy(s, SMPLX_DIR)
assert glob.glob(f'{SMPLX_DIR}/SMPLX_NEUTRAL.*'), \
    "SMPLX_NEUTRAL.npz yok → smpl-x.is.tue.mpg.de, Drive/MeshVTON/smplx/"

# 2) 4D-Humans
try:
    import hmr2  # noqa
except Exception:
    get_ipython().system('pip install -q git+https://github.com/shubham-goel/4D-Humans.git')

# 3) HMR2.0 ağırlıkları (~3.5 GB, bir kez)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)

# 4) SMPL neutral (HMR2 gövde modeli) — smplify.is.tue.mpg.de → Drive'dan
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob('/content/drive/MyDrive/MeshVTON/smpl/*neutral*lbs*.pkl') \
      + glob.glob('/content/drive/MyDrive/MeshVTON/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl yok → smplify.is.tue.mpg.de, Drive/MeshVTON/smpl/"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")

from src.modules.smplx_estimator import RealSMPLXEstimator
estimator = RealSMPLXEstimator(model_path='/content/MeshVTON/checkpoints/pretrained',
                               regressor='hmr2', device='cuda')
estimator.load_model()
print('✅ gerçek SMPL-X tahmincisi hazır (HMR2.0 → SMPL-X)')

## 6️⃣ Kıyafet (.obj) yükle

Kendi `.obj` giysini yükle. Elle yönlendirme yok — giysi her kişide o kişinin pozuna göre
otomatik render edilir.

In [ ]:
from google.colab import files
print("🧵 3D KIYAFET (.obj) yükle:")
MESH_PATH = list(files.upload().keys())[0]
print("kıyafet:", MESH_PATH)

## 7️⃣ (Opsiyonel) Kamera açısını elle geçersiz kıl

Normalde `VIEW_ANGLE=None` → açı, kişinin SMPL-X `global_orient`'inden otomatik gelir
(ön=0°, arka=180°). Tahminci yanlış yön verirse buradan sabit bir açı (örn. `0`, `180`) verebilirsin.

In [ ]:
# None → açı kişinin pozundan otomatik. Sabit istersen: 0 (ön) / 90 (yan) / 180 (arka)
VIEW_ANGLE = None
print("VIEW_ANGLE =", VIEW_ANGLE)

## 8️⃣ Kişileri yükle + Try-On (çoklu kişi, aynı kıyafet)

In [ ]:
from torchvision import transforms
from pathlib import Path
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
device = 'cuda'; dt = torch.float16
H, W = 512, 384   # eğitimle aynı kanonik çözünürlük (parite)

# Kategori (mask bölgesi) — dosya adından
name_lower = MESH_PATH.lower()
if any(k in name_lower for k in ['etek','pantolon','skirt','pants','lower','short','şort','jeans']):
    category = 'lower_body'
elif any(k in name_lower for k in ['elbise','dress','gown']):
    category = 'dresses'
else:
    category = 'upper_body'
print("kategori:", category)

print("👤 KİŞİ görsellerini yükle (birden çok seçebilirsin):")
person_files = list(files.upload().keys())

with torch.no_grad(), torch.amp.autocast('cuda'):
    neg = "monochrome, lowres, bad anatomy, worst quality, low quality"
    pe,npe,ppe,nppe = pipe.encode_prompt("model is wearing clothes",num_images_per_prompt=1,
        do_classifier_free_guidance=True,negative_prompt=neg)
    pe_c,_,_,_ = pipe.encode_prompt(["a photo of clothes"],num_images_per_prompt=1,
        do_classifier_free_guidance=False,negative_prompt=[neg])

results = []
for pf in person_files:
    human = Image.open(pf).convert("RGB").resize((W, H))

    # 3D conditioning — giysiyi KİŞİNİN POZUNA göre render et (eğitimle tek kaynak)
    cond = build_conditioning_3d(human, MESH_PATH, estimator, renderer, draper,
                                 height=H, width=W, view_angle=VIEW_ANGLE,
                                 device=device, dtype=dt)
    cond3d = cond['conditioning_3d']; render_rgb = cond['render_rgb']
    print(f"{pf}: kamera azim = {cond['azim']:.0f}°")

    # 2D ön-işleme (mask + densepose) — kişinin gerçek pozu
    kp = openpose_model(human); parse,_ = parsing_model(human)
    mask,_ = get_mask_location('hd', category, parse, kp); mask = mask.resize((W, H))
    arg = convert_PIL_to_numpy(_apply_exif_orientation(human), format="BGR")
    a = apply_net.create_argument_parser().parse_args(('show',
        '/content/IDM-VTON-official/configs/densepose_rcnn_R_50_FPN_s1x.yaml',
        '/content/IDM-VTON-official/ckpt/densepose/model_final_162be9.pkl',
        'dp_segm','-v','--opts','MODEL.DEVICE','cuda'))
    pose = Image.fromarray(a.func(a, arg)[:, :, ::-1]).resize((W, H))

    with torch.no_grad(), torch.amp.autocast('cuda'):
        res = pipe(prompt_embeds=pe.to(device,dt),negative_prompt_embeds=npe.to(device,dt),
            pooled_prompt_embeds=ppe.to(device,dt),negative_pooled_prompt_embeds=nppe.to(device,dt),
            num_inference_steps=30,generator=torch.Generator(device).manual_seed(42),strength=1.0,
            pose_img=tfm(pose).unsqueeze(0).to(device,dt),text_embeds_cloth=pe_c.to(device,dt),
            cloth=tfm(render_rgb).unsqueeze(0).to(device,dt),mask_image=mask,image=human,
            height=H,width=W,ip_adapter_image=render_rgb,guidance_scale=2.0,
            controlnet_3d=controlnet, conditioning_3d=cond3d)[0][0]
    results.append((human, render_rgb, res))

n = len(results)
fig, ax = plt.subplots(n, 3, figsize=(12, 6*n)); ax = np.atleast_2d(ax)
for i,(h, rr, r) in enumerate(results):
    ax[i,0].imshow(h);  ax[i,0].set_title('Kişi');               ax[i,0].axis('off')
    ax[i,1].imshow(rr); ax[i,1].set_title('3D render (poza göre)'); ax[i,1].axis('off')
    ax[i,2].imshow(r);  ax[i,2].set_title('Sonuç');              ax[i,2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from IPython.display import display
from PIL import Image

n = len(results)
if n > 0:
    print(f"Toplam {n} sonuç gösteriliyor...\n")
    for i, (h, rr, r) in enumerate(results):
        print(f"👤 Kişi {i+1}  |  👕 3D render (poza göre)  |  ✨ Sonuç")
        w, hh = h.width, h.height
        combined = Image.new('RGB', (w * 3, hh))
        combined.paste(h, (0, 0))
        combined.paste(rr.resize((w, hh)), (w, 0))
        combined.paste(r.resize((w, hh)), (w * 2, 0))
        combined.thumbnail((1200, 600))
        display(combined)
        print("-" * 80)
else:
    print("Gösterilecek sonuç yok. Üstteki hücrenin başarıyla tamamlandığından emin ol.")